In [1]:
import pandas as pd
from pathlib import Path
from algo.features import add_labels
from algo.backtester import backtest_ML_switchable, hybrid_entry, default_exit
from algo.model import load_or_train, predict_last    # or your correct import path
from algo.features import add_indicators
import warnings
warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names, but LGBMClassifier was fitted with feature names"
)
# if needed

# === Step 1: Load CSV and set 'date' as datetime index ===
csv_path = Path("data_processed/HDFCBANK_3minute_feat.csv")
df = pd.read_csv(csv_path, parse_dates=["date"], index_col="date")
df = add_labels(df)
df = add_indicators(df)  # <--- All features added here
split_date = '2025-04-01'
df_train = df[df.index < split_date].copy()
df_test  = df[df.index >= split_date].copy()



# 2. Train or load your ML model (adjust path/horizon if needed)
model = load_or_train(df_train, retrain=True, horizon=5)

# 3. Run the backtest
trades, metrics = backtest_ML_switchable(
    df=df_test,
    model=model,
    predict_fn=predict_last,
    entry_rule_fn=hybrid_entry,   # <--- put your rule here!
    exit_rule_fn=default_exit,    # or None to fallback to basic SL/TP/EOD
    capital=350_000,
    contract_size=150,
    lookback=60,      # must match model lookback!
    sl_pct=0.0015,      # 1% stop
    tp_pct=0.0050,      # 2% target
    debug=False
)

# 4. Inspect results:
print(metrics)
print(trades.head())


Prepared 5122 samples | Class balance (mean): 0.513
🔧  Training started …
✅  Finished in 0.2s   (best_iter = 1, best_AUC = 0.5664)
Hold-out accuracy: 0.480
Non-NaN ml_prob: 8815 out of 8875
           ml_prob
count  8815.000000
mean      0.516932
std       0.000000
min       0.516932
25%       0.516932
50%       0.516932
75%       0.516932
max       0.516932
{'Trades': 293, 'WinRate': np.float64(0.2593856655290102), 'GrossPnL': np.float64(12064.440000001186), 'Fees': np.float64(44033.931001244), 'NetPnL': np.float64(-31969.491001242815), 'EquityFinal': np.float64(318030.5089987573)}
             entry_ts             exit_ts side  entry_price   exit_price  \
0 2025-04-01 14:42:00 2025-04-01 15:03:00  BUY      1772.55  1769.891175   
1 2025-04-02 09:15:00 2025-04-02 09:39:00  BUY      1783.15  1792.065750   
2 2025-04-02 09:42:00 2025-04-02 15:12:00  BUY      1789.35  1798.296750   
3 2025-04-02 15:15:00 2025-04-02 15:27:00  BUY      1797.90  1798.900000   
4 2025-04-03 09:45:00 2025-04-

In [2]:
# Ensure entry_ts is a datetime type
trades['entry_ts'] = pd.to_datetime(trades['entry_ts'])

# Extract just the date part
trades['entry_date'] = trades['entry_ts'].dt.date

# Group by date and count
trades_per_day = trades.groupby('entry_date').size().rename('num_trades')

print(trades_per_day)


entry_date
2025-06-16    2
2025-06-17    1
2025-06-18    2
2025-06-19    3
2025-06-20    1
2025-06-23    1
2025-06-24    4
2025-06-25    1
2025-06-26    4
2025-06-27    1
2025-06-30    2
2025-07-01    3
2025-07-02    1
2025-07-03    3
2025-07-04    1
2025-07-07    3
2025-07-08    3
2025-07-09    2
2025-07-10    3
2025-07-11    1
2025-07-14    2
Name: num_trades, dtype: int64


In [1]:
import pandas as pd
from algo.features import add_labels
# load
df = pd.read_csv("data_processed/HDFCBANK_5minute_feat.csv", parse_dates=["date"], index_col="date")
df = add_labels(df, horizon=5)




In [2]:
print(df["label"].value_counts(normalize=True))
# Ideally you want something like 60:40, not 95:5


label
0    0.921321
1    0.078679
Name: proportion, dtype: float64


In [3]:
for col in df.columns:
    if col not in ["label", "date"] and pd.api.types.is_numeric_dtype(df[col]):
        means = df.groupby("label")[col].mean()
        print(f"{col:20s}  mean(0)={means.get(0, float('nan')):.4f}  mean(1)={means.get(1, float('nan')):.4f}")


open                  mean(0)=1602.6077  mean(1)=1568.0766
high                  mean(0)=1603.9424  mean(1)=1569.8834
low                   mean(0)=1601.2181  mean(1)=1566.2665
close                 mean(0)=1602.6140  mean(1)=1568.0969
volume                mean(0)=178678.2629  mean(1)=297398.1191
fees_estimate         mean(0)=28.4256  mean(1)=27.8133
atr                   mean(0)=2.8050  mean(1)=3.3940
atr_median20          mean(0)=2.7992  mean(1)=3.0466
volatility_5          mean(0)=0.0011  mean(1)=0.0015
volatility_10         mean(0)=0.0011  mean(1)=0.0016
minute_of_day         mean(0)=737.9146  mean(1)=763.7697
ret1                  mean(0)=0.0000  mean(1)=-0.0000
ret5                  mean(0)=0.0000  mean(1)=0.0002
fees_pct              mean(0)=0.0177  mean(1)=0.0177
body_1                mean(0)=0.4510  mean(1)=0.5104
below_low_10          mean(0)=0.0672  mean(1)=0.0827
vwap                  mean(0)=1602.5663  mean(1)=1567.8988
close_vs_vwap         mean(0)=0.0000  mean(1)=0.0001

In [4]:
print(df.corr()["label"].sort_values(ascending=False))


label                   1.000000
future_return           0.546720
atr                     0.139421
volatility_5            0.122836
volatility_10           0.121300
volatility              0.095824
zscore_volume           0.084644
trend_strength          0.081337
bb_width                0.079653
vol_spike               0.072514
minute_of_day           0.064310
atr_median20            0.064096
volume                  0.059909
bb_upper_touch          0.040054
body_1                  0.039758
bb_lower_touch          0.024734
below_low_10            0.016513
bb_position             0.013005
zscore_close            0.013005
body_range_ratio        0.012956
rsi_14                  0.012755
ret5                    0.011925
price_vs_vwap           0.006467
vwap_gap                0.006467
close_vs_vwap           0.006368
macd                    0.004770
donchian_breakout       0.001184
is_bullish_engulfing    0.000784
is_hammer              -0.002112
ret1                   -0.002357
macd_signa

In [5]:
from algo.model import load_or_train
model = load_or_train(df, retrain=False, horizon=5)

In [6]:
lgb_model = model.named_steps['lgb']
  # No .named_steps here!

feature_names = [f"f{i}" for i in range(lgb_model.n_features_in_)]
importances = lgb_model.feature_importances_

import pandas as pd
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

print(importance_df.head(20))


     feature  importance
233     f233           5
765     f765           3
423     f423           3
111     f111           3
509     f509           2
461     f461           2
445     f445           2
331     f331           2
574     f574           2
85       f85           2
536     f536           2
750     f750           1
1082   f1082           1
803     f803           1
75       f75           1
522     f522           1
471     f471           1
35       f35           1
255     f255           1
104     f104           1


In [7]:
LOOKBACK = 30
from algo.features import FEATURES
print(f"LOOKBACK: {LOOKBACK}, len(FEATURES): {len(FEATURES)}")
print(FEATURES)


LOOKBACK: 30, len(FEATURES): 34
['ret1', 'ret5', 'atr', 'ema_8', 'ema_21', 'vwap', 'close_vs_vwap', 'body_1', 'rsi_14', 'macd', 'macd_signal', 'supertrend', 'supertrend_dir', 'below_low_10', 'bb_upper', 'bb_lower', 'bb_upper_touch', 'bb_lower_touch', 'bb_width', 'atr_median20', 'zscore_close', 'zscore_volume', 'volatility_5', 'volatility_10', 'vol_spike', 'minute_of_day', 'fees_pct', 'donchian_high', 'donchian_low', 'donchian_breakout', 'is_doji', 'is_hammer', 'is_bullish_engulfing', 'body_range_ratio']


In [8]:
num_feats = len(FEATURES)
lookback = LOOKBACK  # make sure this is 30, or whatever you use

def decode_fidx(idx, num_feats=num_feats, lookback=lookback):
    lookback_idx = idx // num_feats
    feat_idx = idx % num_feats
    t_idx = lookback - lookback_idx - 1  # t-1 is most recent, t-2 next, ...
    return f"t-{t_idx} {FEATURES[feat_idx]}"

for idx in importance_df['feature']:
    fidx = int(str(idx).replace('f',''))
    print(f"Feature: {decode_fidx(fidx)}, Importance: {importance_df.loc[importance_df['feature'] == idx, 'importance'].values[0]}")


Feature: t-23 donchian_breakout, Importance: 5
Feature: t-7 bb_lower_touch, Importance: 3
Feature: t-17 bb_lower, Importance: 3
Feature: t-26 macd, Importance: 3
Feature: t-15 body_range_ratio, Importance: 2
Feature: t-16 atr_median20, Importance: 2
Feature: t-16 ema_8, Importance: 2
Feature: t-20 minute_of_day, Importance: 2
Feature: t-13 is_doji, Importance: 2
Feature: t-27 bb_lower_touch, Importance: 2
Feature: t-14 fees_pct, Importance: 2
Feature: t-7 atr, Importance: 1
Feature: t--2 donchian_low, Importance: 1
Feature: t-6 zscore_volume, Importance: 1
Feature: t-27 body_1, Importance: 1
Feature: t-14 supertrend_dir, Importance: 1
Feature: t-16 donchian_breakout, Importance: 1
Feature: t-28 ret5, Importance: 1
Feature: t-22 bb_lower_touch, Importance: 1
Feature: t-26 atr, Importance: 1
Feature: t--1 vol_spike, Importance: 1
Feature: t-4 vwap, Importance: 1
Feature: t-28 bb_width, Importance: 1
Feature: t-13 bb_upper_touch, Importance: 1
Feature: t-29 bb_lower, Importance: 1
Feature

In [9]:
import pandas as pd
import numpy as np

# --- 1. Set your LOOKBACK and FEATURES exactly as used for training ---
LOOKBACK = 30  # Make sure this matches your model.py!
FEATURES = [
    "ema_21", "bb_lower_touch", "vol_spike", "macd_signal", "bb_lower",
    "below_low_10", "ret1", "vwap", "above_high_10", "bb_lower", "macd",
    "bb_upper", "minute_of_day", "ret5", "ret5", "up_streak", "bb_lower_touch",
    "body_1", "close_vs_vwap", "vwap", "ret5", "macd_signal", "body_1", "close_vs_vwap",
    "vwap", "ret5"
    # ... fill in your full FEATURES list here exactly as used in training!
]
# NOTE: The FEATURES list must match what was used in model training.

# --- 2. Get feature importances from the model ---
lgb_model = model  # or model.named_steps['lgb'] if using pipeline
importances = lgb_model.feature_importances_

# --- 3. Expand features for each lag ---
expanded_features = []
for lag in range(-LOOKBACK, 0):  # Lags: t-30, t-29, ..., t-1, t-0 if LOOKBACK=30
    for feat in FEATURES:
        expanded_features.append(f"t{lag} {feat}")

assert len(expanded_features) == len(importances), (
    f"Mismatch: {len(expanded_features)} features vs {len(importances)} importances"
)

# --- 4. Create DataFrame ---
feature_importances = pd.DataFrame({
    'feature': expanded_features,
    'importance': importances
})

# --- 5. Extract indicator/feature name from the lagged feature string ---
def extract_indicator(feature_name):
    # Example: "t-5 ema_21" -> "ema_21"
    return feature_name.split(" ", 1)[1]

feature_importances['indicator'] = feature_importances['feature'].apply(extract_indicator)
feature_importances['lag'] = feature_importances['feature'].str.extract(r't(-?\d+)').astype(int)

# --- 6. Filter for features with importance >= 10 ---
filtered = feature_importances[feature_importances['importance'] >= 10]

# --- 7. Summarize: group by indicator and show total importance & top lags ---
summary = (
    filtered.groupby('indicator')
    .agg(
        total_importance=('importance', 'sum'),
        top_lags=('lag', lambda lags: list(sorted(lags)))
    )
    .sort_values('total_importance', ascending=False)
)
print(summary)

# Optionally, print the list of features to KEEP (for re-training):
keep_features = filtered['feature'].tolist()
print("Important features to KEEP (for re-training):")
for feat in keep_features:
    print(feat)


AttributeError: 'Pipeline' object has no attribute 'feature_importances_'

DATASET LOADING AND TRAINING

In [16]:
import pandas as pd
from pathlib import Path
from features import add_indicators
# Update filename as needed
csv_path = Path("data/HDFCBANK_5minute.csv")
df_raw = pd.read_csv(csv_path, index_col=0, parse_dates=True).sort_index()
print(f"Loaded {df_raw.shape[0]} rows")


df_feat = add_indicators(df_raw)
print("Indicators added:", list(df_feat.columns))


Loaded 65291 rows
Indicators added: ['open', 'high', 'low', 'close', 'volume', 'atr', 'volatility_5', 'volatility_10', 'minute_of_day', 'ret1', 'ret5', 'body_1', 'below_low_10', 'vwap', 'close_vs_vwap', 'vol_spike', 'ema_8', 'ema_21', 'rsi_14', 'macd', 'macd_signal', 'bb_upper', 'bb_lower', 'bb_upper_touch', 'bb_lower_touch']


In [17]:
from features import add_labels

df_feat = add_labels(df_feat)
print("Labels added. Sample:")
print(df_feat[["label"]].value_counts())


Labels added. Sample:
label
0        60154
1         5137
Name: count, dtype: int64


In [24]:
from model import load_or_train
model = load_or_train(df_feat,retrain=True,horizon=5)


Prepared 9920 samples | Class balance (mean): 0.515
🔧  Training started …
✅  Finished in 0.6s   (best_iter = 7, best_AUC = 0.5347)
Hold-out accuracy: 0.524


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [30]:
import pandas as pd

# Assume: importance_df has columns ['feature', 'importance'], with 'feature' like 'f0', 'f1', ...
# Assume: X is your lagged feature DataFrame with columns ['t-0 macd', ...]
# Assume: FEATURES is your enabled indicators list
# Assume: LOOKBACK is your lookback window (usually 30)

num_feats = len(FEATURES)
lookback = LOOKBACK

def decode_fidx(idx, num_feats=num_feats, lookback=lookback):
    lookback_idx = idx // num_feats
    feat_idx = idx % num_feats
    t_idx = lookback - lookback_idx - 1  # so t-0 is most recent
    return f"t-{t_idx} {FEATURES[feat_idx]}"

# Add decoded names
importance_df['decoded_name'] = importance_df['feature'].apply(
    lambda x: decode_fidx(int(str(x).replace('f','')))
)

# See your top features with decoded names:
print(importance_df[['decoded_name', 'importance']].sort_values('importance', ascending=False).head(20))


           decoded_name  importance
709        t-0 bb_lower           7
716   t-0 price_vs_vwap           6
562     t-6 macd_signal           5
293           t-17 vwap           4
465           t-10 macd           4
205       t-21 bb_lower           3
136   t-24 volatility_5           3
60        t-27 bb_upper           2
269           t-18 vwap           2
393           t-13 macd           2
585            t-5 macd           2
702   t-0 close_vs_vwap           2
571   t-6 minute_of_day           2
172         t-22 ema_21           2
704          t-0 rsi_14           1
281  t-18 volatility_10           1
13        t-29 bb_lower           1
201           t-21 macd           1
197           t-21 vwap           1
706     t-0 macd_signal           1


In [35]:
importance_df['decoded_name'] = importance_df['feature'].apply(
    lambda x: decode_fidx(int(str(x).replace('f','')))
)


In [41]:
# --- Prune helper ---
def make_pruned_df(df_full, importance_df, label_col='label', importance_threshold=0):
    keep_cols = importance_df[importance_df.importance > importance_threshold]['decoded_name'].tolist()
    keep_cols = [c for c in keep_cols if c in df_full.columns]
    return df_full[keep_cols + [label_col]].copy()

# --- Prune and retrain ---
df_pruned = make_pruned_df(df_feat, importance_df, label_col='label', importance_threshold=0)

from model import load_or_train
model = load_or_train(df_pruned, retrain=True)


KeyError: 'close'